# 02 MBA Rules and Recommendations

Compute interpretable two-item market basket rules directly in pandas and overlay customer outcomes and available margin coverage.

In [1]:
from pathlib import Path
import sys


def find_project_root(start=None):
    start = Path.cwd() if start is None else Path(start).resolve()
    for candidate in [start, *start.parents]:
        if (candidate / "EDA" / "outputs").exists() and (candidate / "product_mba").exists():
            return candidate
    raise FileNotFoundError("Could not locate project root containing EDA/outputs and product_mba")

PROJECT_ROOT = find_project_root()
EDA_OUTPUTS = PROJECT_ROOT / "EDA" / "outputs"
MBA_DIR = PROJECT_ROOT / "product_mba"
MBA_OUTPUTS = MBA_DIR / "outputs"
MBA_OUTPUTS.mkdir(exist_ok=True)

print("Project root found")
print("EDA outputs: EDA/outputs")
print("MBA outputs: product_mba/outputs")


Project root found
EDA outputs: EDA/outputs
MBA outputs: product_mba/outputs


In [2]:
import math
import numpy as np
import pandas as pd

customers = pd.read_parquet(EDA_OUTPUTS / "customers.parquet")
try:
    margin_lines = pd.read_parquet(MBA_OUTPUTS / "margin_line_enriched.parquet")
except FileNotFoundError:
    margin_lines = pd.DataFrame()

thresholds = {
    "category": {"min_support": 0.005, "min_confidence": 0.10, "min_lift": 1.10, "min_co_orders": 1, "min_item_orders": 1},
    "handle": {"min_support": 0.003, "min_confidence": 0.08, "min_lift": 1.20, "min_co_orders": 1, "min_item_orders": 1},
    "sku_flavor": {"min_support": 0.0, "min_confidence": 0.0, "min_lift": 1.30, "min_co_orders": 10, "min_item_orders": 50},
}

def build_pair_rules(item_df, item_col, level):
    item_df = item_df[["order_id", "customer_id", item_col]].dropna().drop_duplicates(["order_id", item_col]).copy()
    order_count = item_df["order_id"].nunique()
    item_orders = item_df.groupby(item_col)["order_id"].nunique().rename("antecedent_orders")
    basket_items = item_df.groupby("order_id")[item_col].apply(lambda s: sorted(set(s))).reset_index(name="items")

    pair_rows = []
    for order_id, items in zip(basket_items["order_id"], basket_items["items"]):
        if len(items) < 2:
            continue
        for i, a in enumerate(items):
            for b in items[i + 1:]:
                pair_rows.append((a, b, order_id))
    if not pair_rows:
        return pd.DataFrame()

    pairs = pd.DataFrame(pair_rows, columns=["item_a", "item_b", "order_id"])
    pair_counts = pairs.groupby(["item_a", "item_b"])["order_id"].nunique().reset_index(name="co_orders")

    directional = pd.concat([
        pair_counts.rename(columns={"item_a": "antecedent", "item_b": "consequent"}),
        pair_counts.rename(columns={"item_b": "antecedent", "item_a": "consequent"}),
    ], ignore_index=True)
    directional = directional.merge(item_orders.reset_index().rename(columns={item_col: "antecedent"}), on="antecedent", how="left")
    directional = directional.merge(
        item_orders.reset_index().rename(columns={item_col: "consequent", "antecedent_orders": "consequent_orders"}),
        on="consequent", how="left",
    )
    directional["total_orders"] = order_count
    directional["support"] = directional["co_orders"] / order_count
    directional["antecedent_support"] = directional["antecedent_orders"] / order_count
    directional["consequent_support"] = directional["consequent_orders"] / order_count
    directional["confidence"] = directional["co_orders"] / directional["antecedent_orders"]
    directional["lift"] = directional["confidence"] / directional["consequent_support"]
    directional["leverage"] = directional["support"] - (directional["antecedent_support"] * directional["consequent_support"])
    directional["conviction"] = np.where(
        directional["confidence"] < 1,
        (1 - directional["consequent_support"]) / (1 - directional["confidence"]),
        np.inf,
    )
    directional["level"] = level

    t = thresholds[level]
    filtered = directional[
        (directional["support"] >= t["min_support"])
        & (directional["confidence"] >= t["min_confidence"])
        & (directional["lift"] >= t["min_lift"])
        & (directional["co_orders"] >= t["min_co_orders"])
        & (directional["antecedent_orders"] >= t["min_item_orders"])
        & (directional["consequent_orders"] >= t["min_item_orders"])
    ].copy()
    return filtered.sort_values(["lift", "co_orders", "confidence"], ascending=[False, False, False])

def add_outcomes(rules, item_df, item_col):
    if rules.empty:
        return rules
    cust = customers.copy()
    cust["customer_id"] = cust["customer_id"].astype(str)
    item_df = item_df[["customer_id", item_col]].dropna().drop_duplicates()
    item_df["customer_id"] = item_df["customer_id"].astype(str)

    rows = []
    for idx, row in rules.iterrows():
        customers_with_both = item_df[item_df[item_col].isin([row["antecedent"], row["consequent"]])].groupby("customer_id")[item_col].nunique()
        both_ids = customers_with_both[customers_with_both >= 2].index
        segment = cust[cust["customer_id"].isin(both_ids)]
        rows.append({
            "rule_index": idx,
            "both_item_customers": len(segment),
            "both_item_repeat_rate": segment["is_repeat"].mean() if len(segment) else np.nan,
            "both_item_avg_ltv": segment["total_revenue"].mean() if len(segment) else np.nan,
            "both_item_pct_subscribed": segment["ever_subscribed"].mean() if len(segment) else np.nan,
        })
    outcome = pd.DataFrame(rows).set_index("rule_index")
    return rules.join(outcome)

def add_margin_overlay(rules, level):
    if rules.empty or margin_lines.empty:
        rules["margin_overlay_available"] = False
        return rules
    if level == "category":
        item_col = "product_category"
    elif level == "handle":
        item_col = "Line: Product Handle"
    else:
        item_col = "sku_norm"
    if item_col not in margin_lines.columns:
        rules["margin_overlay_available"] = False
        return rules
    m = (
        margin_lines.dropna(subset=[item_col])
        .groupby(item_col)
        .agg(
            item_revenue_sgd=("net_revenue", "sum"),
            item_covered_revenue_sgd=("net_revenue", lambda s: s[margin_lines.loc[s.index, "has_high_confidence_cost"]].sum()),
            item_est_gross_profit_sgd=("estimated_gross_profit", "sum"),
        )
        .reset_index()
    )
    m["item_margin_coverage_pct"] = np.where(m["item_revenue_sgd"] > 0, m["item_covered_revenue_sgd"] / m["item_revenue_sgd"], np.nan)
    m["item_est_margin_pct"] = np.where(m["item_covered_revenue_sgd"] > 0, m["item_est_gross_profit_sgd"] / m["item_covered_revenue_sgd"], np.nan)
    ant = m.add_prefix("antecedent_").rename(columns={f"antecedent_{item_col}": "antecedent"})
    con = m.add_prefix("consequent_").rename(columns={f"consequent_{item_col}": "consequent"})
    rules = rules.merge(ant, on="antecedent", how="left").merge(con, on="consequent", how="left")
    rules["margin_overlay_available"] = rules[["antecedent_item_margin_coverage_pct", "consequent_item_margin_coverage_pct"]].notna().any(axis=1)
    return rules


In [3]:
rule_tables = {}
for level in ["category", "handle", "sku_flavor"]:
    item_df = pd.read_parquet(MBA_OUTPUTS / f"basket_items_{level}.parquet")
    item_col = item_df.columns[-1]
    rules = build_pair_rules(item_df, item_col, level)
    rules = add_outcomes(rules, item_df, item_col)
    rules = add_margin_overlay(rules, level)
    rule_tables[level] = rules
    rules.to_csv(MBA_OUTPUTS / f"mba_rules_{level}.csv", index=False)
    print(level, "rules:", len(rules))
    display(rules.head(10))

recommendations = []
for level, rules in rule_tables.items():
    if rules.empty:
        continue
    top = rules.sort_values(["lift", "co_orders", "both_item_avg_ltv"], ascending=[False, False, False]).head(15).copy()
    for _, row in top.iterrows():
        recommendations.append({
            "level": level,
            "antecedent": row["antecedent"],
            "consequent": row["consequent"],
            "recommendation_type": "Cross-sell / bundle candidate",
            "rationale": f"{int(row['co_orders'])} co-orders, lift {row['lift']:.2f}, confidence {row['confidence']:.1%}",
            "support": row["support"],
            "confidence": row["confidence"],
            "lift": row["lift"],
            "co_orders": row["co_orders"],
            "avg_ltv_for_both_item_customers": row.get("both_item_avg_ltv", np.nan),
            "repeat_rate_for_both_item_customers": row.get("both_item_repeat_rate", np.nan),
            "margin_overlay_available": row.get("margin_overlay_available", False),
        })
recommendations_df = pd.DataFrame(recommendations)
recommendations_df.to_csv(MBA_OUTPUTS / "mba_recommendations.csv", index=False)
print("Saved recommendations:", len(recommendations_df))
display(recommendations_df.head(20))


category rules: 4


,antecedent,consequent,co_orders,antecedent_orders,consequent_orders,total_orders,support,antecedent_support,consequent_support,confidence,...,antecedent_item_covered_revenue_sgd,antecedent_item_est_gross_profit_sgd,antecedent_item_margin_coverage_pct,antecedent_item_est_margin_pct,consequent_item_revenue_sgd,consequent_item_covered_revenue_sgd,consequent_item_est_gross_profit_sgd,consequent_item_margin_coverage_pct,consequent_item_est_margin_pct,margin_overlay_available
0,Accessories,Lean Protein,774,2275,3760,15000,0.051600,0.151667,0.250667,0.340220,...,23913.295455,19945.915455,0.973852,0.834093,287306.200884,238120.451212,169509.861212,0.828804,0.711866,True
1,Lean Protein,Accessories,774,3760,2275,15000,0.051600,0.250667,0.151667,0.205851,...,238120.451212,169509.861212,0.828804,0.711866,24555.379697,23913.295455,19945.915455,0.973852,0.834093,True
2,Accessories,Clear Protein,734,2275,3728,15000,0.048933,0.151667,0.248533,0.322637,...,23913.295455,19945.915455,0.973852,0.834093,334018.255658,300539.176567,217158.166567,0.899769,0.722562,True
3,Clear Protein,Accessories,734,3728,2275,15000,0.048933,0.248533,0.151667,0.196888,...,300539.176567,217158.166567,0.899769,0.722562,24555.379697,23913.295455,19945.915455,0.973852,0.834093,True


handle rules: 14


,antecedent,consequent,co_orders,antecedent_orders,consequent_orders,total_orders,support,antecedent_support,consequent_support,confidence,...,antecedent_item_covered_revenue_sgd,antecedent_item_est_gross_profit_sgd,antecedent_item_margin_coverage_pct,antecedent_item_est_margin_pct,consequent_item_revenue_sgd,consequent_item_covered_revenue_sgd,consequent_item_est_gross_profit_sgd,consequent_item_margin_coverage_pct,consequent_item_est_margin_pct,margin_overlay_available
0,green-tea-extract-capsules,pureburn-fat-burner-capsules,53,150,248,15000,0.003533,0.010000,0.016533,0.353333,...,3392.469091,2426.829091,0.854375,0.715358,19497.708182,15845.118788,13242.048788,0.812666,0.835718,True
1,pureburn-fat-burner-capsules,green-tea-extract-capsules,53,248,150,15000,0.003533,0.016533,0.010000,0.213710,...,15845.118788,13242.048788,0.812666,0.835718,3970.701515,3392.469091,2426.829091,0.854375,0.715358,True
2,multivitamin-vegan,super-omega-3,50,156,228,15000,0.003333,0.010400,0.015200,0.320513,...,7744.632727,5081.052727,0.805305,0.656074,10225.640000,8415.281515,5490.601515,0.822959,0.652456,True
3,super-omega-3,multivitamin-vegan,50,228,156,15000,0.003333,0.015200,0.010400,0.219298,...,8415.281515,5490.601515,0.822959,0.652456,9617.016364,7744.632727,5081.052727,0.805305,0.656074,True
4,clear-protein-25g-single-sachet,lushprotein-lean-protein-40g-single-serve,230,579,797,15000,0.015333,0.038600,0.053133,0.397237,...,13156.290000,10262.490000,0.962207,0.780044,19780.110000,18874.910000,15369.820000,0.954237,0.814299,True
5,lushprotein-lean-protein-40g-single-serve,clear-protein-25g-single-sachet,230,797,579,15000,0.015333,0.053133,0.038600,0.288582,...,18874.910000,15369.820000,0.954237,0.814299,13673.030000,13156.290000,10262.490000,0.962207,0.780044,True
6,super-omega-3,micronized-creatine-monohydrate,66,228,1419,15000,0.004400,0.015200,0.094600,0.289474,...,8415.281515,5490.601515,0.822959,0.652456,54733.201818,49689.282121,40502.442121,0.907845,0.815114,True
7,discovery-sampler,lushprotein-clear-shaker,136,374,2275,15000,0.009067,0.024933,0.151667,0.363636,...,9081.800000,7346.910000,1.000000,0.808971,24555.379697,23913.295455,19945.915455,0.973852,0.834093,True
8,clear-protein-25g-single-sachet,lushprotein-clear-shaker,206,579,2275,15000,0.013733,0.038600,0.151667,0.355786,...,13156.290000,10262.490000,0.962207,0.780044,24555.379697,23913.295455,19945.915455,0.973852,0.834093,True
9,lushprotein-clear-shaker,clear-protein-25g-single-sachet,206,2275,579,15000,0.013733,0.151667,0.038600,0.090549,...,23913.295455,19945.915455,0.973852,0.834093,13673.030000,13156.290000,10262.490000,0.962207,0.780044,True


sku_flavor rules: 614


,antecedent,consequent,co_orders,antecedent_orders,consequent_orders,total_orders,support,antecedent_support,consequent_support,confidence,...,antecedent_item_covered_revenue_sgd,antecedent_item_est_gross_profit_sgd,antecedent_item_margin_coverage_pct,antecedent_item_est_margin_pct,consequent_item_revenue_sgd,consequent_item_covered_revenue_sgd,consequent_item_est_gross_profit_sgd,consequent_item_margin_coverage_pct,consequent_item_est_margin_pct,margin_overlay_available
0,0724999810494 | Clear Protein Single Serve | 1...,0724999810487 | Clear Protein Single Serve | 1...,71,85,79,15435,0.004600,0.005507,0.005118,0.835294,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,False
1,0724999810487 | Clear Protein Single Serve | 1...,0724999810494 | Clear Protein Single Serve | 1...,71,79,85,15435,0.004600,0.005118,0.005507,0.898734,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,False
2,0724999810494 | Clear Protein Single Serve | 2...,0724999810487 | Clear Protein Single Serve | 2...,97,112,108,15435,0.006284,0.007256,0.006997,0.866071,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,False
3,0724999810487 | Clear Protein Single Serve | 2...,0724999810494 | Clear Protein Single Serve | 2...,97,108,112,15435,0.006284,0.006997,0.007256,0.898148,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,False
4,0724999810470 | Lean Protein Single Serve | 1 ...,0724999810463 | Lean Protein Single Serve | 1 ...,87,104,114,15435,0.005637,0.006738,0.007386,0.836538,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,False
5,0724999810463 | Lean Protein Single Serve | 1 ...,0724999810470 | Lean Protein Single Serve | 1 ...,87,114,104,15435,0.005637,0.007386,0.006738,0.763158,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,False
6,0724999810463 | Lean Protein Single Serve | 40...,0724999810470 | Lean Protein Single Serve | 40...,136,149,158,15435,0.008811,0.009653,0.010236,0.912752,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,False
7,0724999810470 | Lean Protein Single Serve | 40...,0724999810463 | Lean Protein Single Serve | 40...,136,158,149,15435,0.008811,0.010236,0.009653,0.860759,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,False
8,0724999810487 | CLEAR PROTEIN | 25g Sachet (1 ...,0724999810494 | CLEAR PROTEIN | 25g Sachet (1 ...,142,158,158,15435,0.009200,0.010236,0.010236,0.898734,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,False
9,0724999810494 | CLEAR PROTEIN | 25g Sachet (1 ...,0724999810487 | CLEAR PROTEIN | 25g Sachet (1 ...,142,158,158,15435,0.009200,0.010236,0.010236,0.898734,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,False


Saved recommendations: 33


,level,antecedent,consequent,recommendation_type,rationale,support,confidence,lift,co_orders,avg_ltv_for_both_item_customers,repeat_rate_for_both_item_customers,margin_overlay_available
0,category,Accessories,Lean Protein,Cross-sell / bundle candidate,"774 co-orders, lift 1.36, confidence 34.0%",0.051600,0.340220,1.357260,774,250.129228,0.490361,True
1,category,Lean Protein,Accessories,Cross-sell / bundle candidate,"774 co-orders, lift 1.36, confidence 20.6%",0.051600,0.205851,1.357260,774,250.129228,0.490361,True
2,category,Accessories,Clear Protein,Cross-sell / bundle candidate,"734 co-orders, lift 1.30, confidence 32.3%",0.048933,0.322637,1.298165,734,281.165311,0.510843,True
3,category,Clear Protein,Accessories,Cross-sell / bundle candidate,"734 co-orders, lift 1.30, confidence 19.7%",0.048933,0.196888,1.298165,734,281.165311,0.510843,True
4,handle,green-tea-extract-capsules,pureburn-fat-burner-capsules,Cross-sell / bundle candidate,"53 co-orders, lift 21.37, confidence 35.3%",0.003533,0.353333,21.370968,53,1248.505209,0.698113,True
5,handle,pureburn-fat-burner-capsules,green-tea-extract-capsules,Cross-sell / bundle candidate,"53 co-orders, lift 21.37, confidence 21.4%",0.003533,0.213710,21.370968,53,1248.505209,0.698113,True
6,handle,multivitamin-vegan,super-omega-3,Cross-sell / bundle candidate,"50 co-orders, lift 21.09, confidence 32.1%",0.003333,0.320513,21.086370,50,688.447576,0.789474,True
7,handle,super-omega-3,multivitamin-vegan,Cross-sell / bundle candidate,"50 co-orders, lift 21.09, confidence 21.9%",0.003333,0.219298,21.086370,50,688.447576,0.789474,True
8,handle,clear-protein-25g-single-sachet,lushprotein-lean-protein-40g-single-serve,Cross-sell / bundle candidate,"230 co-orders, lift 7.48, confidence 39.7%",0.015333,0.397237,7.476222,230,302.820893,0.400000,True
9,handle,lushprotein-lean-protein-40g-single-serve,clear-protein-25g-single-sachet,Cross-sell / bundle candidate,"230 co-orders, lift 7.48, confidence 28.9%",0.015333,0.288582,7.476222,230,302.820893,0.400000,True


## What The MBA Output Tells Us

The market basket analysis produces three rule tables: category-level rules, product-handle rules, and SKU/flavor rules. A rule like `Accessories -> Lean Protein` means that among orders containing Accessories, Lean Protein appears more often than expected by chance. The most important columns are:

- `co_orders`: how many baskets contained both products.
- `confidence`: of baskets with the antecedent, the share that also contained the consequent.
- `lift`: how much more often the pair occurs together versus random chance. Values above `1.0` indicate positive association.
- `avg_ltv_for_both_item_customers` and `repeat_rate_for_both_item_customers`: whether customers who bought both items look commercially valuable.
- `margin_overlay_available`: whether cost coverage exists for margin-informed decisions. Treat margin as directional because `00_margin_data.ipynb` rated the margin data **usable with caveats**.

### Main Readout

1. Accessories are the clearest category-level cross-sell lever.
   Accessories pair with both Lean Protein and Clear Protein at meaningful scale: `Accessories -> Lean Protein` has 774 co-orders, 34.0% confidence, and 1.36 lift; `Accessories -> Clear Protein` has 734 co-orders, 32.3% confidence, and 1.30 lift.

2. Single-serve sachets behave like variety/discovery products.
   The strongest SKU/flavor rules are mostly Clear Protein Peach with White Grape, and Lean Protein Taro with Thai Milk Tea. These have very high lift because customers often buy these sachets together as flavor trials or bundles.

3. A few supplement pairings are small but very strong.
   Examples include Green Tea Extract with Pureburn Fat Burner, and Multivitamin Vegan with Super Omega-3. These have high lift but lower co-order volume, so they are better treated as targeted bundle tests rather than broad homepage strategy.

4. The recommendations table is not saying every high-lift pair should become a major campaign.
   Prioritize pairs with enough volume (`co_orders`), sensible confidence, and strategic fit. Very high lift with low volume can be useful for niche bundles, but not necessarily for a core growth bet.

### What To Do Exactly

1. Add cart/checkout cross-sell prompts for protein plus accessories.
   When a cart contains Lean Protein or Clear Protein, recommend shakers or relevant accessories. When a cart contains accessories only, recommend Lean Protein first, then Clear Protein.

2. Create or promote variety bundles for sachets.
   Bundle Clear Protein Peach + White Grape, and Lean Protein Taro + Thai Milk Tea. Position these as trial packs, discovery sets, or flavor-comparison bundles.

3. Use targeted supplement bundles, not broad campaigns.
   Test Green Tea Extract + Pureburn Fat Burner, Multivitamin Vegan + Super Omega-3, and Super Omega-3 + Creatine with small placements, email segments, or post-purchase offers.

4. Use `product_mba/outputs/mba_recommendations.csv` as the action list.
   Start with category and handle rows because they are easier to translate into merchandising. Use SKU/flavor rows mainly for flavor bundles and specific product-page recommendations.

5. Do not optimize purely on estimated margin yet.
   Margin coverage is strong for core hero categories but incomplete overall. Use the margin overlay as a guardrail, and ask LushProtein for updated SKU-level COGS/currency/effective-date confirmation before making margin-maximizing decisions.

### Recommended First Experiments

- Experiment 1: Protein + shaker/accessory checkout cross-sell.
- Experiment 2: Clear Protein Peach + White Grape sachet bundle.
- Experiment 3: Lean Protein Taro + Thai Milk Tea sachet bundle.
- Experiment 4: Small targeted supplement bundle test for Green Tea Extract + Pureburn Fat Burner.
- Experiment 5: Post-purchase recommendation email based on first purchased product family.

---
### Final metrics & scores

**Rules retained after thresholds:** category **4** · handle **14** · SKU/flavor **614**.

**Top rules by level** (`co_orders` = baskets containing both items; `lift` > 1 = positive association):

| Level | Rule | Co-orders | Confidence | Lift |
|---|---|--:|--:|--:|
| Category | Accessories ↔ Lean Protein | 774 | 34.0% | 1.36 |
| Category | Accessories ↔ Clear Protein | 734 | 32.3% | 1.30 |
| Handle | clear 25g sachet ↔ lean 40g single-serve | 230 | 39.7% | 7.48 |
| Handle | clear-shaker ↔ lean 40g single-serve | 245 | 30.7% | 2.03 |
| SKU/flavor | Lean Taro ↔ Lean Thai Milk Tea (40g sachets) | 173 | 91–95% | 76.4 |
| SKU/flavor | Clear Peach ↔ Clear White Grape (500g) | 296 | 28–40% | 5.79 |

**Read:** the highest-*volume* cross-category lever is **Accessories ↔ core protein**; the highest-*lift* signals are **flavor-variety sachet pairs** (discovery behaviour). Both feed the retention re-scoring in `03`.
